# NATS-Bench tutorial

NATS-Bench is a **tabular** NAS benchmark: every architecture in the search space has already been trained, so an evaluation is a lookup rather than a training run.

This notebook covers:
1. Installation and the benchmark file
2. Creating the API and querying architectures
3. Random search with an anytime performance plot
4. Learning curves of the top-3 architectures
5. Extras: search-space statistics, validation vs. test, saving results

NATS-Bench has two search spaces: the **topology space (TSS)** with 15,625 cells (identical to NAS-Bench-201), and the **size space (SSS)** with 32,768 width configurations. This tutorial uses TSS. The benchmark covers CIFAR-10, CIFAR-100 and ImageNet16-120.

## 1. Installation and benchmark files

### Install the packages

Run the cell below. It installs `nats_bench` from PyPI, plus the plotting libraries.

### Download the topology benchmark files

NATS-Bench is distributed as data files from the 2020-08-31 release. Download the files for the mode you'll use (see section 3):

| Mode | File | Download |
|---|---|---|
| `FAST_MODE = True` (laptop) | `NATS-tss-v1_0-3ffb9-simple.tar` | [Google Drive](https://drive.google.com/file/d/17_saCsj_krKjlCBLOJEpNtzPXArMCqxU/view?usp=sharing) |
| `FAST_MODE = False` (lab PC) | `NATS-tss-v1_0-3ffb9.pickle.pbz2` | [Google Drive](https://drive.google.com/file/d/1vzyK0UVH2D3fTpa1_dSWnp1gvGpAxRul/view?usp=sharing) |

Steps:
1. **Lab PC:** you can probably keep `NATS-tss-v1_0-3ffb9.pickle.pbz2` compressed. The API reads it directly. **Laptop and smaller PCs:** extract the archive with `tar xvf NATS-tss-v1_0-3ffb9-simple.tar`. This gives a folder named `NATS-tss-v1_0-3ffb9-simple` with one file per architecture, plus a `meta.pickle.pbz2` file. (tested)
3. Save the files in a folder you can find again. The NATS-Bench README suggests `$TORCH_HOME`, where the API finds them automatically. Otherwise give the path explicitly.
4. In section 3, set `BENCH_PATH` to the folder (laptop) or the file (lab PC), and set `FAST_MODE` to match. You can also set the `NATS_BENCH_PATH` environment variable.

If Google Drive blocks the download (it sometimes does for large files), the project README at [github.com/D-X-Y/NATS-Bench](https://github.com/D-X-Y/NATS-Bench) lists mirrors.

This tutorial focusses on the TSS search space

## 2. Installation, only needed first time

In [ ]:
%pip install -q nats_bench matplotlib numpy

## 3. Configuration and Creating the API

`create` loads the benchmark. `FAST_MODE` in section 2 selects one of two modes:

- **`FAST_MODE = True` (recommended for laptop use):** the API reads each architecture's file the first time you query it. Start-up is quick and memory use is low. Requires BENCH_PATH to point to the extracted tar file. 
- **`FAST_MODE = False` (untested, but should work on our lab PCs):** the API loads the whole benchmark into memory at start-up. This takes minutes and several GB of RAM, but every later query is served from memory (making interactions faster). Requires BENCH_PATH to point to the pbz2 file.

In [ ]:
import os
import csv
import random

import numpy as np
import matplotlib.pyplot as plt

from nats_bench import create

SEARCH_SPACES = {'tss': 'NATS-Bench topology (TSS)'} # TODO: expand, there's also 
SEARCH_SPACE = 'tss'
FAST_MODE = True
BENCH_PATH = os.environ.get('NATS_BENCH_PATH', 'NATS-tss-v1_0-3ffb9-simple')

DATASETS = ['cifar10', 'cifar100', 'ImageNet16-120']
DATASET = 'cifar10'

if not os.path.exists(BENCH_PATH):
    raise FileNotFoundError('Benchmark not found at {}. See section 1.'.format(BENCH_PATH))

api = create(BENCH_PATH, SEARCH_SPACE, fast_mode=FAST_MODE, verbose=False)

print('Search space: %s' % SEARCH_SPACES[SEARCH_SPACE])
print('Number of architectures:', len(api))

## 4. Architectures and evaluations

An architecture is a string describing a 4-node cell with 6 edges. Each edge carries one of 5 operations: `none`, `skip_connect`, `nor_conv_1x1`, `nor_conv_3x3`, `avg_pool_3x3`. The format is `|op~0|+|op~0|op~1|+|op~0|op~1|op~2|`: the first group has 1 edge, the second 2 and the third 3.

Architectures are addressed by an integer index, and the API converts between the two.

In [ ]:
arch_str = api.arch(0)
print('Architecture 0:', arch_str)
print('Round-trip index:', api.query_index_by_arch(arch_str))

### Visualising a cell

A cell is a small directed graph. Node 0 is the input and node 3 is the output. Each arrow is an operation applied to the node it starts from, and the node it points to receives the sum of its incoming edges. A dotted `none` edge means no connection, so an architecture is a cell with some edges removed.

`parse_arch` turns the architecture string into a list of edges. It's a good first exercise for students to write themselves.

The network around the cell (stem, three stages of cells, reduction blocks and classifier) is the same for every architecture. The NAS-Bench-201 paper (Dong & Yang, ICLR 2020, arXiv:2001.00326) describes this macro skeleton in detail.

In [ ]:
from matplotlib.patches import Circle, FancyArrowPatch

OP_COLORS = {
    'nor_conv_3x3': '#1f77b4',
    'nor_conv_1x1': '#2ca02c',
    'avg_pool_3x3': '#ff7f0e',
    'skip_connect': '#7f7f7f',
    'none': '#bbbbbb',
}

def parse_arch(arch_str):
    """Turn '|op~0|+|op~0|op~1|+|op~0|op~1|op~2|' into a list of (src, dst, op)."""
    edges = []
    for dst, group in enumerate(arch_str.split('+'), start=1):
        items = group.strip().strip('|').split('|')
        for src, item in enumerate(items):
            edges.append((src, dst, item.split('~')[0]))
    return edges

def draw_cell(arch_str, ax=None, title=None):
    if ax is None:
        _, ax = plt.subplots(figsize=(6, 3))
    for src, dst, op in parse_arch(arch_str):
        color = OP_COLORS.get(op, 'black')
        arrow = FancyArrowPatch((src, 0), (dst, 0),
                                connectionstyle='arc3,rad=-0.6',  # arcs above the baseline
                                arrowstyle='-|>', mutation_scale=14,
                                color=color,
                                linewidth=1 if op == 'none' else 2,
                                linestyle=':' if op == 'none' else '-',
                                shrinkA=10, shrinkB=10)
        ax.add_patch(arrow)
        # With rad=-0.6 the arc peaks at 0.3 * span above the baseline.
        ax.text((src + dst) / 2, 0.3 * (dst - src) + 0.12, op,
                ha='center', va='bottom', fontsize=8, color=color)
    for node in range(4):
        ax.add_patch(Circle((node, 0), 0.12, fc='white', ec='black', zorder=3))
        name = 'input' if node == 0 else ('output' if node == 3 else 'node {}'.format(node))
        ax.text(node, -0.25, name, ha='center', va='top', fontsize=9)
    ax.set_xlim(-0.5, 3.5)
    ax.set_ylim(-0.6, 1.3)
    ax.axis('off')
    if title:
        ax.set_title(title, fontsize=10)
    return ax

The following visualises a given architecture

In [ ]:

arch_idx = 0
arch_str = api.arch(0)
fig, ax = plt.subplots(figsize=(6, 3))
draw_cell(arch_str, ax=ax, title='Architecture %d: %s' % (arch_idx, arch_str))
plt.show()
api.show(arch_idx)

### Querying an evaluation

`get_more_info(index, dataset, hp=..., is_random=False)` returns the results for one architecture on one dataset. `is_random=False` averages over the available training seeds. Setting it to `True` returns a single randomly chosen seed, which mimics one noisy run.

Run the next cell and look at the keys. Key names can differ slightly between NATS-Bench releases, so the helpers below try the common names.

In [ ]:
arch_idx = api.query_index_by_arch(arch_str)
info = api.get_more_info(arch_idx, DATASET, hp=200, is_random=False)
print('Keys:', list(info.keys()))
train_cost = api.get_cost_info(arch_idx, DATASET, hp=200)
print('Keys:', list(train_cost.keys()))

The helpers below wrap the key lookups. `get_final` returns the accuracy at the last epoch. `train_cost` returns the simulated cost of training, which we use as our budget.

## 5. Random search

Random search samples architectures without replacement and evaluates each one. Two rules keep the experiment honest:
- The incumbent (best architecture so far) is chosen by **validation** accuracy only.
- **Test** accuracy is recorded for reporting, and never influences the search.

The simulated cost is the sum of training times of all evaluated architectures, so it's a proxy for compute spent.

In [ ]:
def random_search(seed, hp_schedule, n_evals, dataset):
    records = []
    for idx in range(len(n_evals)):
        # make sure to use the following function:
        # name = 'cifar10-valid' if dataset == 'cifar10' else dataset
        # api.get_more_info(idx, dataset, hp=hp_schedule, is_random=False)
        # note that CIFAR10 has some weird key issues
        # use a debugger to get insights in the proper keys
    return records

In [ ]:
N_CONFIGS = 64
HP_SCHEDULE = 200
N_SEEDS = 10

all_runs = {s: random_search(s, HP_SCHEDULE, N_CONFIGS, DATASET) for s in range(N_SEEDS)}
print('Ran {} seeds x {} evaluations'.format(N_SEEDS, N_CONFIGS))

### Anytime performance plot

Left: test accuracy of the incumbent versus the number of evaluations. Right: the same curve against simulated cost. Shaded bands show ±1 standard deviation across seeds. The two views can disagree, since architectures differ in training cost.

In [ ]:
# make a plot of the random search performance



## 6. Learning curves of the top-3 architectures

Take the three architectures with the best validation accuracy across all seeds, then plot their per-epoch learning curves. Passing `iepoch` to `get_more_info` returns the results at one epoch, so we loop over epochs.

### The top-3 cells

Here are the three architectures from above, drawn as cells.

In [ ]:
def learning_curve(idx, dataset, hp_schedule):
    vals, tests = [], []
    for e in range(200):
        name = 'cifar10-valid' if dataset == 'cifar10' else dataset
        # things get ugly below. note that we call the api twice, and that we need different dataset names for CIFAR-10
        vals.append(api.get_more_info(idx, name, hp=hp_schedule, iepoch=e, is_random=False)['valid-accuracy'])
        tests.append(api.get_more_info(idx, dataset, hp=hp_schedule, iepoch=e, is_random=False)['test-accuracy'])
        
    return np.array(vals), np.array(tests)

# replace 1, 2, 3 with the indexes of the top three architectures

curves = {r: learning_curve(r, DATASET, 200) for r in [1, 2, 3]}
print(curves)

## 7. Final Notes

**Pitfalls**
- **Never select on test accuracy.** The notebook only uses validation accuracy to pick architectures; keep it that way.
- **Noise matters.** Single-seed results (`is_random=True`) vary a lot. Report mean ± std over search seeds, and consider training-seed noise too.
- **Cost is simulated.** Summed training time is a proxy for compute, not your machine's wall-clock time. Check the unit of `train-time` in your release before quoting hours.
- **Key names change between releases.** If a helper raises a `KeyError`, print `info.keys()` and update the candidate names.
- **Validation and test come from different settings.** For CIFAR-10, validation uses `cifar10-valid` and test uses `cifar10`, as in the library's own search protocol. Use `valid_info` for selection and `test_info` only for reporting.
- **Compare like with like.** Use the same `HP` and dataset across methods; 12-epoch and 200-epoch rankings differ.
